In [30]:
import pandas as pd
import numpy as np

import glob
import re
import requests
import io
import os

In [31]:
interaction_data = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interaction_exp.pkl")

In [32]:
interaction_data.head()

,substrate,kinase,p_ind,p_aa
0,P34901,Q05655,183,S
1,Q9UQL6,Q05655,259,S
2,P18433-2,Q05655,204,S
3,P61978,Q05655,302,S
4,P35570,Q05655,612,S


In [33]:
print("number of interactions experimentally validated to be queried")
print(interaction_data.shape[0])

number of interactions experimentally validated to be queried
55398


In [34]:
print("number of unique kinase accession to be queried")
unique_kinases = list(set(interaction_data["kinase"].values))
print(len(unique_kinases))

number of unique kinase accession to be queried
2331


In [35]:
print("number of unique phosphosite accession to be queried")
unique_substrates = list(set(interaction_data["substrate"].values))
print(len(unique_substrates))

number of unique phosphosite accession to be queried
7663


In [36]:
phosphosite_xl = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/pretraining/phosphosite_xl.pkl")
phosxl = list(set(phosphosite_xl["uniprot"]))

In [37]:
substrate_in_xl = [substrate in phosxl for substrate in unique_substrates]

In [38]:
interaction_data_mapped = pd.merge(interaction_data,phosphosite_xl,left_on = "substrate",right_on = "uniprot")

In [39]:
interaction_data_mapped = interaction_data_mapped.loc[:,["substrate","kinase","p_ind","p_aa","sequence"]].drop_duplicates()
interaction_data_mapped

,substrate,kinase,p_ind,p_aa,sequence
0,P34901,Q05655,183,S,MAPVCLFAPLLLLLLGGFPVAPGESIRETEVIDPQDLLEGRYFSGA...
3,P34901,P17252,183,S,MAPVCLFAPLLLLLLGGFPVAPGESIRETEVIDPQDLLEGRYFSGA...
6,P34901,P17252,183,S,MAPVCLFAPLLLLLLGGFPVAPGESIRETEVIDPQDLLEGRYFSGA...
9,P34901,Q05655,183,S,MAPVCLFAPLLLLLLGGFPVAPGESIRETEVIDPQDLLEGRYFSGA...
12,Q9UQL6,Q05655,259,S,MNSPNESDGMSGREPSLEILPRTSLHSIPVTVEVKPVLPRAMPSSM...
...,...,...,...,...,...
2301946,Q91Z79,P20444,760,S,MMCEVMPTISEDGRRGSALGPDDAGGELERLMVTMLTERERLLETL...
2301969,P26048,P05132,359,S,MKTKLSTCNVWSLLLVLLVWDPVRLVLANIQEDEAKNNITIFTRIL...
2301974,O35085,P05132,266,S,MSNQYQEEGCSERPECKSKSPTLLSSYCIDSILGRRSPCKMRLLGA...
2301979,Q8C4Q6,P05132,161,S,MSEVTRSLLQRWGASLRRGADFDSWGQLVEAIDEYQILARHLQKEA...


In [40]:
print("number of unique phosphosite accession mapped now")
mapped_substrates = list(set(interaction_data_mapped["substrate"].values))
print(len(mapped_substrates))

number of unique phosphosite accession mapped now
6287


In [47]:
problem_accs = []
for i in range(interaction_data_mapped.shape[0]):
    s = interaction_data_mapped.iloc[i,:]
    if not ( s["sequence"][int(s["p_ind"])-1] == s["p_aa"]):
        print(s["sequence"])
        print(int(s["p_ind"])-1)
        problem_accs.append(s["substrate"])

MPRYGASLRQSCPRSGREQGQDGTAGAPGLLWMGLVLALALALALALSDSRVLWAPAEAHPLSPQGHPARLHRIVPRLRDVFGWGNLTCPICKGLFTAINLGLKKEPNVARVGSVAIKLCNLLKIAPPAVCQSIVHLFEDDMVEVWRRSVLSPSEACGLLLGSTCGHWDIFSSWNISLPTVPKPPPKPPSPPAPGAPVSRILFLTDLHWDHDYLEGTDPDCADPLCCRRGSGLPPASRPGAGYWGEYSKCDLPLRTLESLLSGLGPAGPFDMVYWTGDIPAHDVWHQTRQDQLRALTTVTALVRKFLGPVPVYPAVGNHESTPVNSFPPPFIEGNHSSRWLYEAMAKAWEPWLPAEALRTLRIGGFYALSPYPGLRLISLNMNFCSRENFWLLINSTDPAGQLQWLVGELQAAEDRGDKVHIIGHIPPGHCLKSWSWNYYRIVARYENTLAAQFFGHTHVDEFEVFYDEETLSRPLAVAFLAPSATTYIGLNPGYRVYQIDGNYSGSSHVVLDHETYILNLTQANIPGAIPHWQLLYRARETYGLPNTLPTAWHNLVYRMRGDMQLFQTFWFLYHKGHPPSEPCGTPCRLATLCAQLSARADSPALCRHLMPDGSLPEAQSLWPRPLFC
509
MPRYGASLRQSCPRSGREQGQDGTAGAPGLLWMGLVLALALALALALSDSRVLWAPAEAHPLSPQGHPARLHRIVPRLRDVFGWGNLTCPICKGLFTAINLGLKKEPNVARVGSVAIKLCNLLKIAPPAVCQSIVHLFEDDMVEVWRRSVLSPSEACGLLLGSTCGHWDIFSSWNISLPTVPKPPPKPPSPPAPGAPVSRILFLTDLHWDHDYLEGTDPDCADPLCCRRGSGLPPASRPGAGYWGEYSKCDLPLRTLESLLSGLGPAGPFDMVYWTGDIPAHDVWHQTRQDQLRALTTVTALVRKFLGPVPVYPAVGNHESTPVNSFPPPFIEGNHSSRWLYEAMAKAWEPWLPAEALRTLRIGGF

MTMDSGADNQQSGDAAVTEAESQQMTVQAQPQIATLAQVSMPAAHATSSAPTVTLVQLPNGQTVQVHGVIQAAQPSVIQSPQVQTVQSSCKDLKRLFSGTQISTIAESEDSQESVDSVTDSQKRREILSRRPSYRKILNDLSSDAPGVPRIEEEKSEEETSAPAITTVTVPTPIYQTSSGQYIAITQGGAIQLANNGTDGVQGLQTLTMTNAAATQPGTTILQYAQTTDGQQILVPSNQVVVQAASGDVQTYQIRTAPTSTIAPGVVMASSPALPTQPAEEAARKREVRLMKNREAARECRRKKKEYVKCLENRVAVLENQNKTLIEELKALKDLYCHKSD
118
MTMDSGADNQQSGDAAVTEAESQQMTVQAQPQIATLAQVSMPAAHATSSAPTVTLVQLPNGQTVQVHGVIQAAQPSVIQSPQVQTVQSSCKDLKRLFSGTQISTIAESEDSQESVDSVTDSQKRREILSRRPSYRKILNDLSSDAPGVPRIEEEKSEEETSAPAITTVTVPTPIYQTSSGQYIAITQGGAIQLANNGTDGVQGLQTLTMTNAAATQPGTTILQYAQTTDGQQILVPSNQVVVQAASGDVQTYQIRTAPTSTIAPGVVMASSPALPTQPAEEAARKREVRLMKNREAARECRRKKKEYVKCLENRVAVLENQNKTLIEELKALKDLYCHKSD
118
MTMDSGADNQQSGDAAVTEAESQQMTVQAQPQIATLAQVSMPAAHATSSAPTVTLVQLPNGQTVQVHGVIQAAQPSVIQSPQVQTVQSSCKDLKRLFSGTQISTIAESEDSQESVDSVTDSQKRREILSRRPSYRKILNDLSSDAPGVPRIEEEKSEEETSAPAITTVTVPTPIYQTSSGQYIAITQGGAIQLANNGTDGVQGLQTLTMTNAAATQPGTTILQYAQTTDGQQILVPSNQVVVQAASGDVQTYQIRTAPTSTIAPGVVMASSPALPTQPAEEAARKREVRLMKNREAARECRRKKKEYV

IndexError: string index out of range

In [63]:
unmap_substrate = [sub for sub in unique_substrates if sub not in phosxl]

In [101]:
sum([str(sub) == 'nan' for sub in unique_substrates])

2

In [5]:
def get_uniprot_entry(acc):
    url = "https://rest.uniprot.org/uniprotkb/search?query="+acc+ "&&format=tsv&fields=id,sequence"
    try:
        urlData = requests.get(url).content
        rawData = pd.read_csv(io.StringIO(urlData.decode('utf-8')),sep = '\t').iloc[0,1]
    except:
        rawData = ""
    return rawData 

In [102]:
# unmap_substrate
res_ls = []
for substrate in unmap_substrate:
    if str(substrate) == 'nan':
        res = ''
    else:
        print(substrate)
        res = get_uniprot_entry(substrate)
    res_ls.append(res)

P17870
P67939
A0A2Z4LIS9
P32850
Q3B891
Q2KIC2
A4K2W5
P06401-2
P18841
Q08E52
P35398-4
Q2QLG7
P24801
Q9QX70
P62978
ENSP00000267569
Q04865
O97952
F1M391
P14100
Q2HJ81
Q6GNV6
Q3T160
Q28589
Q5T7K2
Q8IXJ6-2
Q80ZH1
Q70T02
ABI33873
Q5RAX1
Q5RAG2
Q6QNF3
P50551
Q5EAE6
P0CH28
Q28923
Q6DBX4
Q6NVR1
P41984
Q810W5
ENSP00000248419
A2T767
P16220-1
Q17R91
P04629-2
A4K2S1
Q95168
Q6A548
Q9WVB0-2
Q96IK2
Q9MZD9
Q9GLX8
Q2QL90
Q7YSG7
Q5RDC8
Q28350
A0JNE3
P0CG66
P61142
P62161
Q5R7I1
Q5R4T2
A8E4N3
Q9GKX7
B2RXW8
Q76II0
Q9H4U5
Q5R7D8
A5A6K2
Q9PVX8
P69341
Q91XW0
Q2YDF1
Q8HZP6
A0MLS5
Q64623
Q4R383
O77656
P02541
Q2QLG6
Q3SZB8
Q91636
Q804H7
Q4R4X8
P79394
Q9QUS1
P60952
Q9UC54
P61585
E2RH47
P23355
Q9XT90
Q5RA03
Q864S8
Q8I009
Q2M3B4
Q8MIK0
Q5R943
Q15300
Q8MKI8
Q5TZA2-2
Q6GM86
Q6P7A6
Q6XBN7
Q91783
Q9H2W4
CG6975-PA
A9ULZ2
P81436
Q91431
Q99PW1
Q9XSZ5
D4ACL3
Q7TNI8
Q08499-5
Q7YS81
Q9Y2J6
Q2KJ93
P02718
Q9NH55
Q1JQA2
NP_001005762
A2VDV2
P56951
P10731
A0JNH1
Q5R4K9
Q5R8S0
A5D7P0
Q6FI05
O88944-2
A1YF08
Q9PUM6
Q4R562
P06396-2
C0

Q8HXW3
P29350-3
Q63449
Q4R8C3
P56498
Q4R630
Q3LRT3
P37238-2
Q3T169
Q2EF73
P67938
Q4JF28
Q8TD99
P49841-2
Q60416
P19517
Q05B92
Q28566
P70114
E2RJI4
Q5IS80
ENSMUSP00000098874
D4GYE3
P02313
P41823
O43665-1
P13834
Q28685
Q98TW0
Q7Z5C3
P28053
Q6NTJ3
Q4R633
Q5R9I9
Q4R4R6
Q96PH1-4
P01131
Q8BMH6
Q7ZXH9
Q07E26
P31976
Q9TT90
Q9MYL0
Q9DGD3
B2CXA1
Q5R5A0
Q9MZ06
Q3ZBL9
P46633
P34815-2
P20460
Q60550
A1YG85
P11309-2
Q08DC7
P78348-1
D3YN49
Q6J6I8
XP_849482
Q7YR76
Q92126
P41110
Q8JH64
P62139
Q5R8T7
Q99MM3
E2RK09
P62991
O35800
Q13878
A4K2Y1
Q6PQZ2
Q5RBU9
Q08493-2
NP_001258431
Q5R4F3
P51662
A7X8B7
Q9XT09
Q28521
NP_001099736
O53380
Q920M9
P29353-3
Q5VT18
O46469
Q3MHP5
P26630
Q3ZBU7
Q9N261
P0CG55
P28025
Q60524
CG4005-PB
Q62985-2
Q5R902
Q08289-2
Q9MZL5
Q9DE07
Q9GJP9
C7GVI6
Q92021
P17290
Q2T9U2
Q28317
Q06124-1
B5KFD7
P37176
P17970
Q108U7
Q5R9Y6
Q6URC2
Q764M6
O02755
Q1WK23
Q60495
P62801
Q6PI20
P05786
Q5FBR4
O46606
P11845
Q07E25
Q50L39
O97600
A5A6L0
P10829
F1MRK3
P63058_OBS
A2T6Y4
Q75M87
F1SRY5
Q66WT7
Q5RD00
O7

In [104]:
len(res_ls) == len(unmap_substrate)

True

In [111]:
phospho_seq = pd.concat([phosphosite_xl.loc[:,["uniprot","sequence"]].drop_duplicates(), \
pd.DataFrame({"uniprot":unmap_substrate,"sequence":res_ls}).iloc[2:,:]],axis = 0)

In [112]:
interaction_data_mapped = pd.merge(interaction_data,phospho_seq,left_on = "substrate",right_on = "uniprot")

In [113]:
print("number of unique phosphosite accession mapped now")
mapped_substrates = list(set(interaction_data_mapped["substrate"].values))
print(len(mapped_substrates))

number of unique phosphosite accession mapped now
7661


In [115]:
interaction_data_mapped.columns = ["substrate","kinase","p_ind","p_aa","uniprot","substrate_seq"]
interaction_data_mapped = interaction_data_mapped.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq"]]

In [116]:
interaction_data_mapped.head()

,substrate,kinase,p_ind,p_aa,substrate_seq
0,P34901,Q05655,183,S,MAPVCLFAPLLLLLLGGFPVAPGESIRETEVIDPQDLLEGRYFSGA...
1,P34901,P17252,183,S,MAPVCLFAPLLLLLLGGFPVAPGESIRETEVIDPQDLLEGRYFSGA...
2,P34901,P17252,183,S,MAPVCLFAPLLLLLLGGFPVAPGESIRETEVIDPQDLLEGRYFSGA...
3,P34901,Q05655,183,S,MAPVCLFAPLLLLLLGGFPVAPGESIRETEVIDPQDLLEGRYFSGA...
4,Q9UQL6,Q05655,259,S,MNSPNESDGMSGREPSLEILPRTSLHSIPVTVEVKPVLPRAMPSSM...


## Fetch kinase sequence

In [117]:
kinase_xl = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/pretraining/kinase_xl.pkl")

In [148]:
interaction_data_mapped["p_ind"] = pd.to_numeric(interaction_data_mapped["p_ind"])
interaction_data_mapped = interaction_data_mapped.drop_duplicates()

In [149]:
res_df = pd.merge(interaction_data_mapped,kinase_xl,left_on = "kinase",right_on = "Entry")

In [127]:
kinase_xl.loc[kinase_xl["Entry"] == "Q05655",:]

,Entry,Entry name,Status,Protein names,Gene names,Organism,Sequence,Length,Cross-reference (Pfam),Cross-reference (RCSB-PDB),Cross-reference (AlphaFoldDB),Cross-reference (PROSITE),Domain [CC],Domain [FT],Protein families,subfamily
449,Q05655,KPCD_HUMAN,reviewed,Protein kinase C delta type (EC 2.7.11.13) (Ty...,PRKCD PKCD,Homo sapiens (Human),MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,676,PF00130;PF00069;PF00433;,NaN,Q05655;,PS51285;PS50004;PS00107;PS50011;PS00108;PS0047...,"DOMAIN: The C1 domain, containing the phorbol ...","DOMAIN 1..106; /note=""C2""; /evidence=""ECO:00...","Protein kinase superfamily, AGC Ser/Thr protei...",AGC Ser/Thr protein kinase family


In [156]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df["kinase"])))

number of mapped kinase out of 2331 kinases 
565


In [200]:
dataraw = "${SPARK_ACQUISITION_ROOT_ALT}/protein_databases"
filename = "uniprot_kinase_prosite.tab"
filepath = os.path.join(dataraw,filename)
uniprot_kinase_prosite = pd.read_csv(filepath, \
                    sep = "\t")

In [201]:
res_df2 = pd.merge(interaction_data_mapped,uniprot_kinase_prosite,left_on = "kinase",right_on = "Entry")

In [202]:
res_df12 = pd.concat([res_df.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq","Sequence", "Organism","Domain [CC]","Domain [FT]","Protein families"]],\
res_df2.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq","Sequence", "Organism","Domain [CC]","Domain [FT]","Protein families"]]],axis = 0).drop_duplicates()

In [203]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df2["kinase"])))

number of mapped kinase out of 2331 kinases 
2040


In [204]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df12["kinase"])))

number of mapped kinase out of 2331 kinases 
2040


In [205]:
dataraw = "${SPARK_ACQUISITION_ROOT_ALT}/protein_databases"
filename = "uniprot_kinase_interpro.tab"
filepath = os.path.join(dataraw,filename)
uniprot_kinase_interpro = pd.read_csv(filepath, \
                    sep = "\t")
res_df3 = pd.merge(interaction_data_mapped,uniprot_kinase_interpro,left_on = "kinase",right_on = "Entry")
res_df123 = pd.concat([res_df3.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq","Sequence", "Organism","Domain [CC]","Domain [FT]","Protein families"]],\
res_df12.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq","Sequence", "Organism","Domain [CC]","Domain [FT]","Protein families"]]],axis = 0).drop_duplicates()

In [206]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df3["kinase"])))

number of mapped kinase out of 2331 kinases 
2134


In [207]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df123["kinase"])))

number of mapped kinase out of 2331 kinases 
2134


In [208]:
dataraw = "${SPARK_ACQUISITION_ROOT_ALT}/protein_databases"
filename = "uniprot_kinase_supfam.tab"
filepath = os.path.join(dataraw,filename)
uniprot_kinase_supfam = pd.read_csv(filepath, \
                    sep = "\t")
res_df4 = pd.merge(interaction_data_mapped,uniprot_kinase_supfam,left_on = "kinase",right_on = "Entry")
res_df124 = pd.concat([res_df4.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq","Sequence", "Organism","Domain [CC]","Domain [FT]","Protein families"]],\
res_df123.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq","Sequence", "Organism","Domain [CC]","Domain [FT]","Protein families"]]],axis = 0).drop_duplicates()

In [209]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df4["kinase"])))

number of mapped kinase out of 2331 kinases 
2134


In [210]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df124["kinase"])))

number of mapped kinase out of 2331 kinases 
2134


In [212]:
uniprot_kinase_interpro.head(1)

,Unnamed: 0,Entry,Entry name,Status,Protein names,Gene names,Organism,Sequence,Length,Cross-reference (Pfam),Cross-reference (RCSB-PDB),Cross-reference (AlphaFoldDB),Cross-reference (Interpro),Domain [CC],Domain [FT],Protein families
0,0,Q6PHZ2,KCC2D_MOUSE,reviewed,Calcium/calmodulin-dependent protein kinase ty...,Camk2d Kiaa4163,Mus musculus (Mouse),MASTTTCTRFTDEYQLFEELGKGAFSVVRRCMKIPTGQEYAAKIIN...,499,PF08332;PF00069;,NaN,Q6PHZ2;,IPR013543;IPR011009;IPR032710;IPR000719;IPR017...,DOMAIN: The CAMK2 protein kinases contain a un...,"DOMAIN 14..272; /note=""Protein kinase""; /evi...","Protein kinase superfamily, CAMK Ser/Thr prote..."


In [182]:
dataraw = "${SPARK_ACQUISITION_ROOT_ALT}/protein_databases"
filename = "uniprot_kinase_interpro_fam20c.tab"
filepath = os.path.join(dataraw,filename)
uniprot_kinase_interpro_fam20c = pd.read_csv(filepath, \
                    sep = "\t")
res_df5 = pd.merge(interaction_data_mapped,uniprot_kinase_interpro_fam20c,left_on = "kinase",right_on = "Entry")
res_df12345 = pd.concat([res_df5.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq","Sequence", "Organism","Domain [CC]","Domain [FT]","Protein families"]],\
res_df124.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq","Sequence", "Organism","Domain [CC]","Domain [FT]","Protein families"]]],axis = 0).drop_duplicates()

In [183]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df5["kinase"])))

number of mapped kinase out of 2331 kinases 
5


In [184]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df12345["kinase"])))

number of mapped kinase out of 2331 kinases 
2139


In [185]:
dataraw = "${SPARK_ACQUISITION_ROOT_ALT}/protein_databases"
filename = "uniprot_kinase_interpro_histadine.tab"
filepath = os.path.join(dataraw,filename)
uniprot_kinase_interpro_histadine = pd.read_csv(filepath, \
                    sep = "\t")
res_df6 = pd.merge(interaction_data_mapped,uniprot_kinase_interpro_histadine,left_on = "kinase",right_on = "Entry")
res_df123456 = pd.concat([res_df6.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq","Sequence"]],\
res_df12345.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq","Sequence"]]],axis = 0).drop_duplicates()

/usr/local/lib/python3.8/dist-packages/IPython/core/interactiveshell.py:3135: DtypeWarning: Columns (11,13) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [186]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df6["kinase"])))

number of mapped kinase out of 2331 kinases 
34


In [187]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df123456["kinase"])))

number of mapped kinase out of 2331 kinases 
2173


In [194]:
unmapped_kinase = interaction_data_mapped["kinase"].drop_duplicates()[~interaction_data_mapped["kinase"].drop_duplicates().isin(res_df123456["kinase"])]

In [198]:
uniprot_kinase_interpro_histadine.loc[:,[]]

,Organism,Domain [CC],Domain [FT],Protein families
0,Staphylococcus aureus (strain MRSA252),NaN,"DOMAIN 61..114; /note=""HAMP""; /evidence=""ECO...",NaN
1,Bacillus subtilis (strain 168),NaN,"DOMAIN 189..405; /note=""Histidine kinase""; /...",NaN
2,Pseudomonas aeruginosa (strain ATCC 15692 / DS...,NaN,"DOMAIN 217..269; /note=""HAMP""; /evidence=""EC...",NaN
3,Aneurinibacillus migulanus (Bacillus migulanus),NaN,"DOMAIN 66..118; /note=""HAMP""; /evidence=""ECO...",NaN
4,Staphylococcus aureus (strain Newman),NaN,"DOMAIN 186..238; /note=""HAMP""; /evidence=""EC...",NaN
...,...,...,...,...
1244081,Rhodanobacter sp. C05,NaN,"DOMAIN 76..128; /note=""PAC""; /evidence=""ECO:...",NaN
1244082,Usitatibacter rugosus,NaN,"DOMAIN 167..385; /note=""Histidine kinase""; /...",NaN
1244083,Paenibacillus peoriae,NaN,"DOMAIN 227..448; /note=""Histidine kinase""; /...",NaN
1244084,Bacillus sp. FDAARGOS_527,NaN,"DOMAIN 223..269; /note=""HAMP""; /evidence=""EC...",NaN


## final mapping of kinase using interpro information

In [214]:
dataraw = "${SPARK_ACQUISITION_ROOT_ALT}/protein_databases"
filename = "uniprot_kinase_interpro.tab"
filepath = os.path.join(dataraw,filename)
uniprot_kinase_interpro = pd.read_csv(filepath, \
                    sep = "\t")
res_df1 = pd.merge(interaction_data_mapped,uniprot_kinase_interpro,left_on = "kinase",right_on = "Entry")
res_df1["kinase_flag"] = "pk"

filename = "uniprot_kinase_interpro_fam20c.tab"
filepath = os.path.join(dataraw,filename)
uniprot_kinase_interpro_fam20c = pd.read_csv(filepath, \
                    sep = "\t")
res_df2 = pd.merge(interaction_data_mapped,uniprot_kinase_interpro_fam20c,left_on = "kinase",right_on = "Entry")
res_df2["kinase_flag"] = "fam20c"

filename = "uniprot_kinase_interpro_histadine.tab"
filepath = os.path.join(dataraw,filename)
uniprot_kinase_interpro_histadine = pd.read_csv(filepath, \
                    sep = "\t")
res_df3 = pd.merge(interaction_data_mapped,uniprot_kinase_interpro_histadine,left_on = "kinase",right_on = "Entry")
res_df3["kinase_flag"] = "histadine"

/usr/local/lib/python3.8/dist-packages/IPython/core/interactiveshell.py:3135: DtypeWarning: Columns (11,13) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [216]:
res_df = pd.concat([res_df1,res_df2,res_df3])

In [217]:
print("number of mapped kinase out of 2331 kinases ")
print(sum(interaction_data_mapped["kinase"].drop_duplicates().isin(res_df["kinase"])))

number of mapped kinase out of 2331 kinases 
2173


In [219]:
res_df = res_df.loc[:,["substrate","kinase","p_ind","p_aa","substrate_seq","Sequence","Cross-reference (Pfam)","Cross-reference (Interpro)","Domain [CC]","Domain [FT]","Protein families","kinase_flag"]]

In [222]:
res_df.shape == res_df.drop_duplicates().shape

True

In [224]:
res_df.columns = ["substrate","kinase","p_ind","p_aa","substrate_seq","kinase_seq","kinase_pfam","kinase_interpro","domain_cc","domain_ft","protein_families","kinase_flag"]

In [262]:
def get_kinase_domain_ind(s):
    s = str(s)
    try:
        res = [doc.split(";")[0].split("..") for doc in s.split("DOMAIN ")[1:] if "Protein kinase" in doc]
    except:
        res = []
    if len(res) == 0:
        try:
            res = [doc.split(";")[0].split("..") for doc in s.split("DOMAIN ")[1:] if "Histidine kinase" in doc]
        except:
            res = []
    if len(res) == 0:
        try:
            res = [doc.split(";")[0].split("..") for doc in s.split("DOMAIN ")[1:] if "Fam20C" in doc]
        except:
            res = []
    return res

In [263]:
kinase_coord = res_df["domain_ft"].apply(lambda s: get_kinase_domain_ind(s))

In [266]:
res_df["kinase_coord"] = kinase_coord

In [281]:
res_df["kinase_coord"].apply(lambda s: len(s)==1).sum()

30752

In [279]:
res_df["kinase_coord"].apply(lambda s: len(s)<1).sum()

1696

In [277]:
res_df["kinase_coord"].apply(lambda s: len(s)>1).sum()

808

In [282]:
res_df = res_df.loc[res_df["kinase_coord"].apply(lambda s: len(s)==1),:]

In [283]:
res_df.shape

(30752, 13)

In [284]:
dataraw

'${SPARK_ACQUISITION_ROOT_ALT}/protein_databases'

In [285]:
res_df.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interaction_exp_cleaned_220915.pkl")

In [101]:
dat = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interaction_exp_cleaned_220915.pkl")

In [102]:
res_ls = []
for substrate in dat['substrate']:
    if str(substrate) == 'nan':
        res = ''
    else:
        print(substrate)
        res = get_uniprot_entry(substrate)
    res_ls.append(res)

P34901
Q9UQL6
P18433-2
P18433-2
P61978
P61978
P35570
P35570
P35570
P53004
P53004
P53004
P53004
P15941
P35611
P13789
P13789
P13789
P13789
P14598
P14598
P14598
P14598
P14598
P14598
P14598
P14598
P14598
Q16206
O35303
O70468
P19429
P19429
P19429
P15407
P15407
O76090
Q9UBS0
Q9H257
O46469
P05107
P05107
Q05655
Q05655
Q05655
Q05655
Q05655
Q05655
Q05655
Q05655
P13796
Q9HBA0
Q9HBA0
Q9HBA0
P05787
P04406
P04406
Q02297-6
P51398
P51398
P51398
P51398
P51398
P42224
P37230
P37230
P37230
P62753
P62753
Q8WN94
Q8WN94
Q99808
Q14247
P98082
Q9Y2R2
Q15139
Q15139
Q15139
P29473
P15381
P15381
P15381
P15381
Q8K585
Q8K585
P08057
P08057
P08057
P08057
P08057
P08057
Q60795
Q08775
P17405
P17405
P68431
P28329-3
P28329-3
P13498
P04637
Q8TCU6
P78536
Q6AYD3
P12830
P35568
P35568
P35568
P35568
P21730
Q9NRY6
Q96A00
P11413-3
P11413-3
P53671
P10588
P70181
P40189
P23396
P23396
Q80X41
O14939
O95817
P29353-2
Q15717
Q15717
Q62318
P35222
P61073
P61073
P49840
Q96C90
P30273
Q61143
Q9BZK7
P40763
P49841
Q9WVE0
O00429
Q09472
Q01543
P085

P19490
P08670
Q01970
P18052
P18052
O14713
P23818
P47709
Q96PH1-4
Q96PH1-4
Q96PH1-4
Q96PH1-4
Q96PH1-4
P19493
P31645
Q9QYP1
Q9QYP1
Q6SLE7
Q99K10
O08553
P15337
P15337
P07293
Q9QUH6
Q9CQD6
P21333
B0LPN4
B0LPN4
P16070
O43526-4
Q00960
Q14674
C6K2K4
P07101-3
P07101-3
P31000
Q06210
B2DD29
O14490-2
P29476
Q13224
P29172
P19020
P17599
P10522
P10522
Q66JT0
P42261
P09917
Q04760
P37840
P10636-8
P10636-8
Q9WV18
P08485
P61810
P55055
P62198
Q9Z297
P51790
P51790
P34152-3
Q00613
P26678
E9Q401
E9Q401
O43524
P00533
P00533
P00533
P00533
P00533
P04626
Q14457
Q14457
Q14457
Q9UQM7-2
Q01147
O95180
Q03555
Q03555
Q12879
P78352
P70313
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
Q14524
O75469
P47712
P49654
Q9NQC7
Q9NQC7
Q9NQC7
O95997
O95997
O95997
O95997
Q9BZB8
P34152
F6SEU4
F6SEU4
F6SEU4
F6SEU4
F6SEU4
F6SEU4
F6SEU4
F6SEU4
F6SEU4
P

P97826
O08589
O08589
Q10728
Q10728
O15217
P10636-2
P10636-2
P10636-2
Q9EPK8
P00523
P47736
P47736
Q05209
Q05209
Q08495
Q62101
Q62101
P17612
P17612
P17612
P17612
P17612
P17612
P17612
P17612
P17612
P17612
Q13522
Q13522
Q13522
Q63943
Q63943
P15146-3
P15146-3
P15146-3
O95180
P54296
Q9XSM3
Q920N7
P22091
Q9Y233-2
Q5S006
Q9UD71
Q00756
O15554
Q14847
Q3UVX5
P17655
P17655
P35560-2
P35560-2
P35560-2
Q95N27
Q99259
P20942
P20942
Q07955
Q8CGN5
Q8CGN5
Q02952
Q02952
Q02952
Q02952
O43164
O43164
P18828
O00482
O94811
O94811
O94811
Q91Y55
Q96T88
Q00059
Q00059
Q00059
Q00059
P30306
P30306
P30306
Q99LM3
Q7TNL5
Q7TNL5
Q7TNL5
Q7TNL5
Q9UH77
Q9Y4C1
P04176
Q60875
Q60875
Q8WZ42
Q8WZ42
Q8WZ42
Q01728
Q92917
Q92917
O60479
P07953
Q08499-6
Q99NE5-7
Q71RJ2
P26955
P81122
P81122
P02687
P02687
P02687
P02687
P02687
P02687
P02687
P02687
Q09470
Q9GZU1
Q9GZU1
Q62600
Q62600
Q62600
Q63664
Q63664
P01103
Q92063
P62195
Q7TQN4
P53805-2
P29994
P29994
P41181
P04083
Q8N5S9
Q8N5S9
O94763
Q08493-2
O08773
O08773
NP_001076191
O75688
P54253


Q3KR16
Q9UBB4
Q9Y266
Q13315
Q9NQG5
O95235
P33778
P33778
Q53HL2
Q53HL2
Q53HL2
Q53HL2
Q53HL2
P98179
Q8NEM2
Q99879
Q99879
Q99879
Q99879
Q93079
Q93079
Q93079
Q93079
P58876
P58876
P58876
P58876
P23527
P23527
P23527
P23527
Q6DN03
Q6DN03
Q6DN03
Q6DN03
Q17R91
Q17R91
Q17R91
Q17R91
P35659
P62807
P62807
P62807
P62807
Q9H410
Q9H410
O95239
O95239
Q53SF7
Q53SF7
Q8NEY8
P05204
P05204
Q08J23
P18615
Q02241-2
Q9H6F5
Q16778
Q16778
Q16778
Q16778
P46778
Q9H788
Q8NG31
Q8NG31
Q69YH5
Q69YH5
Q69YH5
Q96CF2
Q96CF2
Q96CF2
O60814
O60814
O60814
O60814
Q99877
Q99877
Q99877
Q99877
Q8WYJ6
Q8WYJ6
Q8WYJ6
Q8WYJ6
Q8WYJ6
Q8WYJ6
P46783
Q9Y2R4
P06899
P06899
P06899
P06899
Q6DRA6
Q6DRA6
Q6DRA6
Q6DRA6
Q5T5U3
Q5T5U3
P45973
Q5QNW6
Q5QNW6
Q5QNW6
Q5QNW6
Q8IVT2
Q9P227
Q6NZI2
Q9Y5B9
P62917
Q9BW61
Q9NP77
Q9NP77
Q9NP77
O00566
Q9NQZ2
O15523
Q8TE77
Q96E39
Q6LED0
O76021
Q71DI3
Q71DI3
Q9NZD0
Q9NZD0
Q9NZD0
O95229
O95229
O95229
Q9CSU0
Q96K21
Q8TD99
Q8TD99
Q8TD99
Q6NWP9
Q6NWP9
Q16695
Q9P2W8
Q8N850
Q96KS4
Q96KS4
Q96KS4
Q9NYP9
Q92995
P20263
P295

O75581
O60716
O60716
O60716
P42227
P42229
Q01844
P04150
P42225
Q92934
Q02750
Q02750
Q969V6
P07101-2
P49815
Q15648
Q15648
P49790
P49790
P49790
P49790
P49790
P49790
P49790
P49790
P49790
P49790
Q92974
P33076
P33076
P54227
P55211
Q86UR1
P61586
P61586
P21146
Q01860
Q08499-2
O95644
Q9UJY1
Q9UJY1
Q9UJY1
Q05209
Q3UVX5
Q3UVX5
P17655
Q08499-6
P27708
P17676
P08955
P17542
P06239
P24529
Q15746
Q15746
Q15746
P16144
Q9R0Z9
P19793
P36956
Q07820
Q07820
P49715
P23928
Q16828
Q16828
Q16828
Q16828
Q9DBB1
Q9DBB1
P17679
P17679
Q16690
Q16690
Q16690
P15336
P15336
P49023-2
P49023-2
P49023-2
Q63531
Q63531
Q63531
P37285
Q9BUB5
P28033
P28033
P28033
Q05682
Q05682
P70302
P70302
P70302
Q9NYA1
Q9H8Y8
Q9H8Y8
Q8WX93
Q8WX93
Q9EPI6
O70337
P50616
P50616
P50616
P10415
P10415
P10415
Q60793
P37238
Q9NQ66
Q13950
Q13950
Q13950
Q13950
Q13950
Q13950
Q13950
Q13950
Q13950
P04972
O75030-9
Q13614
Q9Z1S8
Q9Z1S8
Q9Z1S8
Q9Z1S8
Q9Z1S8
Q6U1J6
Q80YV2
Q80YV2
P41212
P41212
P19419
P19419
P62813
P12980
P08047
P08047
P08047
O43561-2
Q15788
Q157

...

O35618
O35618
O08696
P48964
P48964
P48964
P48754
P61290
P02340
Q14934
Q14934
P23443
O00327
O00327
O00327
Q8K120
Q8K120
O54918
O54918
O54918
Q60700
Q60700
P27661
P15037
P46938
P46938
P46938
P84244
P20263
Q9WVI9
Q9WVI9
P28700
P28700
P28700
P28700
Q99KG5-2
Q9Z131
B2RS91
P02340
P10415
P10415
P10415
P10415
P97287
P97287
O54918
O54918
O54918
Q60700
Q60700
P05627
P28738
P12023
Q9WVI9
Q9WVI9
Q9ESN9-2
Q9ESN9-2
Q9ESN9-2
P02340
P02340
P02340
P39689
P39689
Q9DBR7
Q9DBR7
Q9DBR7
O08738
Q8BYR2
P02340
P02340
P02340
P16951
P38432
O54962
O54962
O54962
Q5SU73
P02340
O54962
O54962
O54962
Q5SU73
O70343
Q01147
Q04887
Q9Z1S8
Q9Z1S8
Q9R1E0
Q9R1E0
O35099
Q8CFH6
O35491
Q99104
P97474
Q60949
Q9Z1T6
Q91V92
Q61409
P35821
Q9WV06
Q7TPS5
Q9WV89
O43765
O35326
P70460
P20152
P31001
P31001
P31001
P26645
Q9Z207
Q9Z207
Q9DBR7
Q9DAW9
Q9DAW9
Q9ESN9
Q9ESN9
Q9CQ19
O54784
O08586
O08586
O08586
Q99NH2
P03995
P03995
P03995
Q922U2
P62962
P53668
P12023-2
O54785
Q6P9Q4
P70460
Q99LM3
E9PZQ0
Q9DBR7
Q9DBR7
P23819
Q9JMB0
P55088
Q64213
Q8B

P32298
P32298
P32298
P32298
P32298
P32298
P32298
P32298
P13863
P13863
Q16832
Q16832
Q16832
Q13322
Q13322
P05480
Q9QWY8
Q8VI36
Q9QVP9
O88351
O88351
Q9QY53
Q9QY53
Q9WUN2
Q8CHT1
Q00535
P54764
P54764
P54764
P10686
P10686
P05696
P68403
Q4KM52
Q4KM52
Q4KM52
Q4KM52
Q4KM52
P63319
Q920L0
O70601
P10686
P10686
Q63604
Q63604
Q63604
Q63604
Q63604
P13595
P13596
P10686
P0C0K5
P10686
Q60598
Q60598
P28867
P28867
Q04736
Q04736
P42866
P42866
P42866
P42866
P42866
P42866
P42866
P42866
P33534
P33534
P33534
P56726
P56726
P56726
P56726
P56726
P56726
P56726
P26040
Q62406
Q07139
Q6P1M3
Q15334
Q3TIR3
P23528
P45592
P60981
Q9Y281
P23528
P45592
P60981
P23528
P23528
P23528
P45592
P23528
P23528
P60981
P18760
Q63572
P37173
P37173
P37173
P37173
P37173
P37173
P36897
P36897
P36897
P36897
P36897
P36897
P17813
P17813
O75604
O75604
O35346
O08816
Q63767
Q63767
Q63767
Q9Z1P2
Q9QZC5
Q9QZC5
Q66HL2
Q66HL2
O35346
G3V9H8
G3V9H8
G3V9H8
G3V9H8
Q91YM2
Q921Q7
Q4JIM5
Q4JIM5
Q4JIM5
Q4JIM5
P05622
P05622
P05622
Q9Z2U0
O54926
O00401
Q9QUI0

O16844
O16844
P22813
Q01842
P22813
Q01842
P18432
P18432
P40423
P19107
P19107
P08630
P19538
P19538
P19538
CG6975-PA
CG6975-PA
Q95V55
Q95V55
Q95V55
Q24119
Q24119
Q9VW83
Q9VW83
Q10666
Q10666
CG6757-PA
Q9NCC3
P34707
P34707
Q8IPH9
P45594
CG3981-PB
Q9W525
CG4005-PB
Q45VV3
Q5E9F8
Q5E9F8
P84227
P84227
P68432
P68432
Q6J1J1
Q5E9F8
P84227
Q3SZB8
A5PK61
P68432
A5D7P0
Q2HJ21
A7MBD1
A7MBD1
A7MBD1
A7MBD1
A7MBD1
A7MBD1
Q2KJ94
P27925
A5PK26
Q08E52
Q3T169
Q2YDF1
Q32PF2
Q24K09
E1BPQ1
E1BPQ1
E1BPQ1
Q95KV1
A7E3S4
Q08DI6
Q5E9P3
Q5E9L6
O97790
O97790
O97790
Q27ID4
P67808
P27925
E1BPQ1
E1BPQ1
E1BPQ1
Q27994
P46196
Q3SYX0
Q3SYX0
Q3SYX0
Q3SYX0
Q3SYX0
P27925
P62157
P27925
Q32PF2
E1BPQ1
P27925
Q0VCX4
Q0VCX4
Q08DA8
P50148
Q62910-5
Q62910-5
Q62910-5
P97588
P97588
P97588
P02544
Q00366
Q00366
Q00366
P02544
P02544
P02544
Q00366
P37882
P02541
P02541
P02541
P48968
P48968
O09181
P02544
Q00366
Q00366
Q00366
P02541
P02541
P02541
O88529
O88529
Q60528
Q60528
Q60550
Q60528
Q60528
Q60528
C0HJG9
O88479
O88479
Q0QF17
P20460
P20460

P08034
P29319
P29319
P29319
P11627
P11627
A2VDV2
P22328
A0MLS5
A0MLS5
Q8MJ26
Q8MJ26
Q8MJ26
Q8MJ26
Q8MJ26
Q8MJ26
Q8MJ26
Q8MJ26
Q8MJ26
P56424
B8XIA5
Q8MJ26
G3V735
Q63065
Q63065
Q63065
Q63827
Q63827
Q63827
Q63827
Q63827
Q63827
Q63827
Q63065
A1YB07
P39743
P39743
P39743
Q02399
P79132
Q28824
Q28824
Q28824
P63243
Q2KJ94
Q5E9Y2
Q02399
Q3I5G7
Q3I5G7
Q03114
P58772
P49139
P10830
P10102
P05772
P10829
O19111
P67998
Q9XT18
Q6TYA7
Q6TYA7
Q08DJ8
Q3ZBT0
Q3T149
Q3T149
Q3T149
A6QQQ9
P53781
P53781
P69341
Q08DJ8
Q08DJ8
Q3ZBT0
Q32KW2
Q08DJ8
O77627
O77627
Q3ZBT0
O02718
O02718
O02718
A7YY54
Q0VBZ9
Q0VBZ9
Q0VBZ9
Q5EAD3
Q5E9A1
Q9BMK9
Q9BMK9
Q9SIZ2
Q9FE78
Q38997
P92958
Q38997
P92958
P15409
P15409
Q91ZZ3
Q9Z0F7
P51682
P51682
P51682
P51682
P25023
P25023
O08565
O08565
O08565
O08565
Q5XFN2
Q5XFN2
Q5XFN2
E2RH47
Q6SLL5
Q865S3
Q8I009
E2RJI4
Q29537
Q5XFN2
Q5XFN2
Q5XFN2
Q8I009
Q29537
Q29537
Q29537
Q921Y0
Q8BYR2
Q8BYR2
O35942
O35942
Q8VHJ5
Q05512
P52785
P52785
P52785
P52785
P52785
P52785
P52785
P52785
Q28055
P68210
Q5ZLY8

P61145
Q8AYC9
Q6BBL6
Q8QGQ6
Q8QGQ6
A8X6H4
P37803
P37803
P37803
Q9GK38
Q9GK38
Q9GK38
Q9GK38
Q9GK38
Q08E26
Q0IHT3
Q9XSI1
P42072
Q6TYA8
Q6TYA8
Q29101
Q0Z8U2
Q0Z8U2
A6XKM2
A6XKM2
P25930
P25930
P25930
P25930
Q8HZU1
Q8HZU1
Q8HZU1
Q8HZU1
Q3LSL6
Q3LSL6
Q3LSL6
Q3LSL6
O62747
O62747
O62747
O62747
P56498
P56498
P56498
P56498
Q28474
Q28474
Q28474
Q28474
P79394
P79394
P79394
P79394
P61072
P61072
P61072
P61072
P56491
P56491
P56491
P56491
Q6P730
P14607
P14607
P02541
P02541
Q3MHP5
P62629
Q66RN5
Q66RN5
A2Q0Z0
A2Q0Z0
Q5R1X2
Q5R1X2
A2T756
Q5R4R8
Q5R7C0
Q5R7C0
P68105
A0SXL6
A0SXL6
A0SXL6
P49032
Q5R8Z3
Q5R8Z3
P55245
P55245
P81623
P81623
O77656
Q9GLE4
P28053
P52555
P52555
P23097
Q10739
Q9TV98
Q9TV98
P79892
Q8HZQ5
Q95209
Q8HZQ5
Q5RF31
C8BKD0
Q28566
P51664
P51664
P03969
Q60487
P48798
Q5IS69
P48799
P13109
Q9Z2Q8
Q9Z2Q8
Q9Z2Q8
Q9Z2Q8
Q63980
Q9Z2Q8
Q9Z2Q8
Q9Z2Q8
Q9Z2Q8
Q8HZP6
Q8HZP6
Q8HZP6
Q8HZP6
O02761
O02761
Q6S9E0
O02761
O02761
Q810W5
Q810W5
Q810W5
Q810W5
Q810W5
Q810W5
Q810W5
Q810W5
Q810W5
Q2EF73
Q810W5
A4L7N3

In [103]:
len(res_ls)

30752

In [104]:
dat['substrate_seq'] = res_ls

In [105]:
def get_uniprot_entry_2(acc):
    url = "https://legacy.uniprot.org/uniparc/?query="+acc+ "&format=tab&columns=id,sequence"
    try:
        urlData = requests.get(url).content
        rawData = pd.read_csv(io.StringIO(urlData.decode('utf-8')),sep = '\t').iloc[0,1]
    except:
        rawData = ""
    return rawData 

In [106]:
get_uniprot_entry_2("P18433-2")

'MDSWFILVLLGSGLICVSANNATTVAPSVGITRLINSSTAEPVKEEAKTSNPTSSLTSLSVAPTFSPNITLGPTYLTTVNSSDSDNGTTRTASTNSIGITISPNGTWLPDNQFTDARTEPWEGNSSTAATTPETFPPSVALSSLLVIVFIIIVLYMLRFKKYKQAGSHSNSKQAGSHSNSFRLSNGRTEDVEPQSVPLLARSPSTNRKYPPLPVDKLEEEINRRMADDNKLFREEFNALPACPIQATCEAASKEENKEKNRYVNILPYDHSRVHLTPVEGVPDSDYINASFINGYQEKNKFIAAQGPKEETVNDFWRMIWEQNTATIVMVTNLKERKECKCAQYWPDQGCWTYGNIRVSVEDVTVLVDYTVRKFCIQQVGDMTNRKPQRLITQFHFTSWPDFGVPFTPIGMLKFLKKVKACNPQYAGAIVVHCSAGVGRTGTFVVIDAMLDMMHTERKVDVYGFVSRIRAQRCQMVQTDMQYVFIYQALLEHYLYGDTELEVTSLETHLQKIYNKIPGTSNNGLEEEFKKLTSIKIQNDKMRTGNLPANMKKNRVLQIIPYEFNRVIIPVKRGEENTDYVNASFIDGYRQKDSYIASQGPLLHTIEDFWRMIWEWKSCSIVMLTELEERGQEKCAQYWPSDGLVSYGDITVELKKEEECESYTVRDLLVTNTRENKSRQIRQFHFHGWPEVGIPSDGKGMISIIAAVQKQQQQSGNHPITVHCSAGAGRTGTFCALSTVLERVKAEGILDVFQTVKSLRLQRPHMVQTLEQYEFCYKVVQEYIDAFSDYANFK'

In [107]:
for i in range(len(res_ls)):
    if res_ls[i] == '':
        print(i)
        res_ls[i] = get_uniprot_entry_2(dat['substrate'].iloc[i])

2
3
830
873
973
2438
2510
2702
2703
2704
2705
2726
2727
2776
3697
4896
4949
4950
4951
4952
5677
5683
5785
7057
7058
7059
7060
8260
8261
8575
8576
8742
8743
8744
8745
8746
10012
10013
10014
10160
10861
11741
12307
13307
13936
15754
15755
16071
16092
16093
16125
17467
17468
17469
19005
20033
20067
20068
20069
20070
20800
21007
21008
21488
21712
22957
22958
22959
26960
27588
27589
27605
27606
27616
27622
27624


In [108]:
dat['substrate_seq'] = res_ls

In [109]:
wierd_accs = []
nan_accs = []
unmap_accs = []
for i in range(dat.shape[0]):
    s = dat.iloc[i,:]
    print(len(str(s["substrate_seq"])))
    print(s["p_ind"]-1)
    if (str(s["substrate_seq"]) == 'nan'):
        nan_accs.append(i)
    else:
        try:
            is_valid = s["substrate_seq"][s["p_ind"]-1] == s["p_aa"]
            if not is_valid:
                unmap_accs.append(i)
        except:
            wierd_accs.append(i)
            

202
182
1122
258
793
203
793
179
463
301
463
283
1235
611
1235
306
1235
23
296
236
296
229
296
32
296
20
1255
1223
737
725
285
190
285
199
285
280
285
194
390
369
390
302
390
327
390
319
390
358
390
303
390
314
390
378
390
347
610
503
755
634
1270
301
210
142
210
22
210
23
271
216
271
226
585
357
482
472
536
230
484
474
769
757
769
744
676
217
676
502
676
298
676
301
676
644
676
303
676
294
676
140
627
4
871
161
871
174
871
188
483
73
335
245
335
240
645
285
398
30
398
279
398
236
398
184
398
185
750
726
468
128
468
109
468
178
249
234
249
235
87
64
87
41
456
280
550
404
770
23
807
34
912
741
912
737
912
411
1205
1178
2171
1673
2171
1927
2171
527
2171
532
107
43
107
63
212
23
212
78
212
43
212
144
212
45
212
24
597
39
607
325
631
507
631
509
136
44
630
475
630
439
195
146
393
45
1659
312
824
734
394
359
882
789
1242
573
1242
322
1242
306
1242
440
350
333
295
20
147
37
545
209
545
265
638
282
404
82
539
412
918
889
243
5
243
220
440
354
933
565
575
186
473
28
326
317
326
220
834
472
781

468
128
468
162
912
420
2171
1927
2171
1699
2171
1626
212
23
212
43
212
24
212
45
393
377
1659
435
147
37
781
674
781
44
781
551
483
20
86
68
930
27
930
68
420
8
198
26
198
37
736
636
536
16
536
418
204
154
204
111
204
135
555
41
555
55
555
62
2139
1896
52
15
1906
139
1906
1761
1906
148
1906
1748
630
551
466
6
487
397
382
367
382
368
382
364
382
372
2169
1573
308
272
244
70
501
43
701
688
619
6
2336
2125
149
62
149
15
1068
862
716
418
716
419
716
442
716
297
716
445
716
298
716
290
528
70
508
158
4967
2030
4967
2807
907
862
466
72
466
38
466
71
1234
1104
907
862
684
233
684
273
902
861
327
128
327
132
1873
1574
2647
2151
4953
2803
4953
2797
742
696
841
454
841
437
2120
1500
525
408
497
39
699
204
699
234
1484
1165
555
14
906
862
674
523
674
271
674
447
674
670
441
213
441
355
441
319
441
261
441
244
441
323
441
315
441
292
441
408
441
415
441
304
529
319
52
15
4966
2029
4966
2806
2364
2158
595
235
595
304
332
154
478
396
142
26
142
82
864
424
864
410
327
132
327
118
628
620
732
89
551


465
130
393
291
1871
1674
1512
885
1512
998
726
700
726
603
726
616
664
391
928
794
928
806
928
810
826
686
496
94
2527
1626
149
37
149
24
335
190
204
74
219
159
219
13
219
17
498
30
1244
730
350
91
557
243
505
295
505
272
505
111
768
282
668
649
668
452
501
251
365
144
1317
33
1247
356
1247
299
864
777
1356
228
591
84
591
243
858
519
858
606
858
655
858
804
1294
540
330
319
270
28
270
32
270
22
770
742
652
245
240
73
473
395
1616
153
887
667
887
158
1528
641
1528
945
1528
556
1528
858
3142
1198
3142
431
3142
1178
1531
134
497
136
467
353
507
429
346
154
346
221
572
521
570
521
605
394
605
438
605
432
846
450
846
460
458
399
502
410
502
428
502
423
267
60
496
39
496
162
496
120
367
30
367
8
931
13
180
67
180
72
505
272
543
20
864
779
521
443
317
231
382
281
382
278
307
7
307
137
443
320
555
490
555
399
716
502
716
473
1503
1449
1333
221
369
326
369
160
572
508
497
136
54
12
401
394
401
396
649
141
171
66
171
5
926
50
926
394
365
144
559
440
559
309
559
396
677
26
677
626
2964
1571
1092

736
615
2414
2038
2414
1037
531
36
164
129
164
97
1906
1767
630
615
630
606
630
601
467
254
467
7
467
249
467
244
467
219
501
34
619
52
149
24
149
37
528
61
198
186
630
615
497
30
441
230
441
201
441
198
441
216
441
49
441
395
441
403
441
180
441
174
441
211
441
152
441
204
441
421
441
234
441
45
529
306
529
325
802
359
802
580
802
699
595
117
595
105
595
103
331
72
331
62
498
350
498
342
498
345
1332
143
409
13
409
275
1531
1360
1531
1246
1531
1353
1531
1392
1531
1212
640
531
651
632
439
61
202
67
477
71
1464
735
464
61
118
64
1215
1034
1215
1030
933
293
772
342
772
567
217
150
1600
1231
198
68
1113
547
1113
541
920
650
920
515
648
288
648
300
648
295
298
159
462
202
454
78
673
293
673
343
673
424
673
11
1210
692
1335
695
1335
862
1335
7
474
147
474
146
474
312
474
261
474
137
474
468
474
165
474
94
474
466
474
141
474
138
474
142
474
166
474
460
474
150
474
264
1255
700
766
400
766
752
766
150
367
358
367
363
1613
1489
1613
1571
968
905
968
309
968
307
770
726
794
779
656
78
777
225


3
615
3
119
3
68
900
7
900
10
900
726
453
440
453
2
943
733
349
268
694
667
376
231
219
208
290
6
377
231
647
498
463
215
463
283
550
404
393
314
393
391
326
201
770
726
736
615
394
33
429
174
429
18
164
129
467
254
467
249
467
244
149
24
149
37
198
186
441
230
441
201
441
204
595
117
595
105
595
103
595
293
595
340
741
48
2089
1629
2089
340
2089
1067
651
43
480
476
480
478
904
40
904
12
904
107
904
39
904
52
904
138
904
26
1863
1496
1863
113
1863
966
439
61
391
327
882
517
1230
435
587
408
683
166
473
213
933
675
933
212
933
399
933
161
933
189
933
293
919
90
919
50
919
232
919
75
700
486
700
493
700
576
700
443
700
439
700
519
572
508
2115
1775
636
85
747
71
920
95
140
18
298
159
782
508
389
367
655
248
655
297
2843
1359
480
205
931
572
595
590
1910
1000
1910
968
1417
174
476
285
476
300
524
282
3256
356
306
134
706
422
1581
1206
1581
1214
474
243
808
721
609
454
380
186
1060
879
928
355
928
825
928
820
928
372
928
251
928
607
928
806
928
566
928
248
928
810
928
611
496
137
335
227
7

...

209
85
265
135
319
315
319
177
319
79
319
81
319
51
319
84
319
249
319
248
319
175
319
263
319
57
319
265
319
104
319
83
431
12
330
242
656
335
656
320
296
187
168
80
168
77
305
83
525
120
338
53
338
91
338
202
641
240
423
51
647
322
624
542
305
83
716
509
318
183
3
242
439
113
1056
1032
525
411
1648
55
1648
1219
1648
1216
1648
606
1648
1218
868
552
172
18
172
19
172
18
172
19
171
19
171
18
1586
148
172
18
172
19
136
10
172
19
1442
307
1482
1302
172
19
172
18
172
19
448
303
448
305
448
306
448
312
448
320
249
234
249
235
448
116
165
72
288
187
480
307
471
375
480
307
473
181
587
195
732
390
732
623
724
88
1585
1155
732
37
297
14
298
13
298
14
479
182
297
14
732
262
117
98
117
110
735
656
724
254
724
225
401
76
401
74
149
117
149
101
149
79
149
81
429
173
429
387
429
386
458
142
1938
2
1938
1
551
535
1613
1478
2843
1506
2843
1504
2843
1509
2843
1503
512
12
721
543
735
640
2555
2510
208
193
353
13
353
64
419
270
419
274
393
291
1438
311
1210
677
117
63
117
99
334
245
572
521
716
538
439


1255
589
2097
317
1091
454
1100
272
432
49
328
67
1016
196
557
98
313
304
269
85
375
152
466
71
469
16
469
75
469
76
309
151
1171
1069
1171
1060
1029
693
330
292
330
295
1337
314
1337
364
172
19
448
264
403
379
403
381
403
382
1333
829
430
6
430
11
430
34
580
159
140
137
647
507
695
654
638
504
1197
1173
375
234
459
300
5035
2843
1029
689
1029
692
883
716
366
105
323
110
653
19
552
171
559
182
559
182
374
176
385
179
458
181
1424
856
473
181
721
188
156
17
362
59
781
44
195
60
305
130
305
129
781
674
619
519
619
282
498
30
393
217
393
221
358
182
358
184
380
204
380
202
358
184
358
182
380
202
380
204
236
69
722
538
1023
15
1023
22
586
566
722
207
793
214
543
112
297
188
292
124
292
197
920
559
391
312
391
389
3343
3221
329
155
270
22
1163
15
858
594
600
108
600
110
380
186
439
61
923
398
425
7
425
178
425
212
543
112
920
812
543
112
938
203
918
915
875
872
875
710
938
177
692
170
512
226
793
59
1581
1463
938
177
873
296
873
391
419
172
959
483
1585
1155
776
185
1134
330
1134
331
1134


887
248
472
299
472
383
472
349
472
463
630
214
985
142
985
894
540
173
540
175
540
177
540
180
540
473
540
179
504
452
504
428
512
507
601
380
601
448
654
358
654
378
654
354
654
340
654
351
658
639
658
653
658
648
658
645
658
645
658
648
658
653
658
639
1130
734
409
138
309
151
309
162
309
155
175
44
175
58
681
270
681
233
1380
844
801
276
488
381
488
111
465
280
959
634
645
306
677
513
428
239
1033
131
210
208
977
735
977
594
977
772
1033
724
465
321
52
50
315
51
315
48
315
51
52
47
315
51
315
48
1114
560
1114
614
315
51
1114
979
315
51
315
48
315
51
976
771
976
920
976
959
976
574
976
693
976
587
976
593
976
627
976
734
209
207
543
425
168
162
366
46
692
170
692
170
1113
249
1113
487
619
161
619
369
619
61
619
328
509
495
1076
244
1076
464
1134
330
1134
331
1134
394
747
140
747
135
102
50
102
8
486
215
486
79
711
547
770
756
304
220
968
346
434
240
292
14
875
438
875
717
774
34
469
372
1097
684
1097
932
1097
968
178
13
317
51
301
155
733
126
1004
370
422
385
422
201
527
324
527
184

395
119
395
128
313
217
132
121
261
186
261
188
172
119
188
76
191
77
770
121
770
435
770
112
770
136
335
237
335
242
335
241
361
51
572
373
572
543
566
174
566
262
819
35
819
101
819
110
819
117
819
130
819
155
819
168
819
224
819
253
819
279
819
378
819
394
819
437
819
609
819
628
819
687
819
44
819
311
819
195
819
110
819
675
819
384
819
35
819
123
819
427
819
372
819
135
819
200
819
283
819
120
819
383
819
265
819
132
819
366
819
261
819
322
819
378
819
126
819
73
819
293
819
684
819
448
819
62
819
609
819
69
819
214
819
104
819
55
819
117
819
101
819
428
819
60
819
253
819
262
819
663
819
691
819
344
819
183
361
1
361
13
361
16
361
20
361
41
361
49
361
51
361
54
361
73
361
90
361
114
361
128
361
138
361
176
361
190
361
191
361
267
1522
16
1522
236
1522
3
910
674
910
889
314
29
314
270
821
773
821
174
821
374
284
44
284
32
284
47
284
68
284
75
284
4
284
79
819
110
819
135
819
311
819
378
819
394
819
437
819
687
1270
356
904
39
904
52
295
244
427
126
659
381
659
425
659
635
294
3
52

896
354
896
384
896
510
896
527
896
891
257
67
210
3
136
3
136
3
136
6
136
11
136
6
136
11
136
3
136
3
136
3
1111
249
1111
488
1133
330
1133
331
1133
395
1122
258
1122
497
1122
258
1122
497
1122
258
1122
497
393
8
393
14
393
32
393
45
393
14
393
391
393
17
393
19
1863
986
393
19
393
32
393
36
393
45
393
45
393
54
393
182
393
268
393
283
393
314
1863
306
393
314
444
171
393
314
393
391
439
61
261
210
52
15
52
15
844
14
317
303
698
538
154
38
154
82
154
38
154
71
297
169
297
174
297
179
297
183
297
258
154
55
382
14
382
32
154
71
382
170
382
257
382
272
500
31
500
34
500
43
500
45
500
46
500
47
770
742
770
756
770
742
770
756
317
51
770
742
318
232
466
55
770
742
770
756
178
13
770
742
770
756
178
13
317
51
330
260
330
262
330
269
480
317
330
260
330
262
330
269
385
280
385
288
385
132
552
171
554
177
552
171
554
354
554
362
554
391
554
480
554
482
517
354
440
402
481
308
402
73
972
945
766
106
907
66
907
271
907
365
912
68
912
232
912
276
912
314
912
353
912
364
912
369
912
370
912
400


In [110]:
len(nan_accs)

376

In [111]:
len(unmap_accs)

458

In [112]:
len(wierd_accs)

20

In [113]:
dat = dat.reset_index().iloc[:,1:]

In [114]:
dat['valid_check'] = "valid"
dat.loc[nan_accs,'valid_check'] = "nan"
dat.loc[unmap_accs,'valid_check'] = "not_match"
dat.loc[wierd_accs,'valid_check'] = "cannot_index"

In [115]:
dat_valid = dat.loc[dat['valid_check'] == "valid",:]

In [116]:
dat_invalid = dat.loc[dat['valid_check'] != "valid",:]

In [117]:
dat_valid.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interaction_exp_cleaned_220921.pkl")

In [118]:
dat_invalid.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interaction_exp_unmapped_220921.pkl")

In [119]:
dat = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interaction_exp_cleaned_220921.pkl")

In [120]:
for i in range(dat.shape[0]):
    s = dat.iloc[i,:]
#     subseq = s["substrate_seq"][start_ind:end_ind]
    kinase_coord = s["kinase_coord"]
    try: 
        kinseq = s["kinase_seq"][int(kinase_coord[0][0])-1:int(kinase_coord[0][1])]
    except:
        print(i)

28306
28307


In [121]:
dat.loc[29158,"kinase_coord"][0][0] = '1'

In [122]:
dat.loc[29159,"kinase_coord"][0][0] = '1'

In [123]:
dat.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interaction_exp_cleaned_220923.pkl")

In [100]:
# https://rest.uniprot.org/uniparc/search?query=P18433-2&format=tsv&fields=id,sequence
# "https://legacy.uniprot.org/uniparc/?query="+"P13789"+ "&format=tab&columns=id,sequence"